Instalação do Pydantic

In [1]:
!pip install pydantic
!pip install pydantic[email]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 9.8 MB/s eta 0:00:00


Importações das bibliotecas necessárias

In [2]:
from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_validator, field_serializer, model_validator, model_serializer, computed_field, UUID4

In [3]:
app = FastAPI()

Criação classe User (Usuário)

In [17]:
class User(BaseModel):
    model_config = {
        "extra": "forbid",  # Proíbe a adição de atributos extras
    }
    __users__ = []
    name: str = Field(..., min_length=2, max_length=80, description="Name of the user")
    email: EmailStr = Field(..., description="Email address of the user")  # Valida e-mail automaticamente
    peso_kg: float = Field(..., gt=0, description="Peso em kilogramas")
    altura_cm: float = Field(..., gt=0, description="Altura em centimetros")
    sexo: str = Field(..., description="Sexo: M or F")
    idade: int = Field(..., ge=0, description="Idade em anos")
    signup_ts: Optional[datetime] = Field(
        default_factory=datetime.now, description="Signup timestamp", kw_only=True
    )
    id: UUID4 = Field(
        default_factory=uuid4, description="Unique identifier", kw_only=True
    )

    @field_validator("sexo") # Valida se sexo possui valores compatíveis
    @classmethod
    def validate_sexo(cls, v: str) -> str:
        v = v.strip().upper()
        if v not in {"M", "F"}:
            raise ValueError("sexo deve ser 'M' ou 'F'")
        return v

    @field_validator("peso_kg") # Valida se o peso está entre um intervalo plausível
    @classmethod
    def validate_peso(cls, v: float) -> float:
        if not (20 <= v <= 400):
            raise ValueError("peso_kg deve ser entre 20 e 400")
        return float(v)

    @field_validator("altura_cm") # Valida se altura está entre um intervalo plausível
    @classmethod
    def validate_altura(cls, v: float) -> float:
        if not (100 <= v <= 250):
            raise ValueError("altura_cm deve estar entre 100 e 250")
        return float(v)

    @field_validator("idade") # Valida se idade está entre um intervalo plausível
    @classmethod
    def validate_idade(cls, v: int) -> int:
        if not (10 <= v <= 120):
            raise ValueError("idade deve estar entre 10 e 120")
        return int(v)

    @model_validator(mode="after")
    def validate_imc_plausible(self): # Valida se IMC está em uma faixa plausível
        imc = self.imc
        if not (10 <= imc <= 80):
            raise ValueError("imc está fora de um valor plausível (verifique peso/altura)")
        return self

    @computed_field(return_type=float) # Define um campo calculado
    @property # permite acessar como atributo
    def imc(self) -> float:
        altura_m = self.altura_cm / 100
        return self.peso_kg / (altura_m * altura_m)

    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)

    @field_serializer("imc", when_used="json")
    def serialize_imc(self, imc: float) -> float:
        # Arredonda IMC no JSON
        return round(imc, 2)

    @field_serializer("peso_kg", when_used="json")
    def serialize_peso(self, w: float) -> float:
        # Arredonda Peso no JSON
        return round(w, 1)


Criação de Endpoints

In [18]:
@app.get("/users", response_model=list[User]) # Cria um endpoint, sendo a resposta uma lista de usuários (Validada e serializada pelo modelo 'User')
async def get_users() -> list[User]:
    return list(User.__users__)


@app.post("/users", response_model=User) # Cria um endpoint, a reposta será um único objeto User
async def create_user(user: User):
    User.__users__.append(user) # Adiciona o novo usuário à lista em memória
    return user


@app.get("/users/{user_id}", response_model=User) # Cria um endpoint
async def get_user(user_id: UUID4) -> User | JSONResponse: # Recebe o user_id da URL, o FastAPI valida automaticamente se é um UUID válido
    try:
        return next((user for user in User.__users__ if user.id == user_id)) # Retorna o primeiro usuário cujo id seja igual ao user_id
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "User not found"})

Criação da função principal

In [19]:
def main() -> None:
    with TestClient(app) as client:  # Cria um cliente de teste do FastAPI
        for i in range(5):  # Cria 5 usuários
            payload = {
                "name": f"User {i}",
                "email": f"example{i}@arjancodes.com",
                "peso_kg": 70 + i,
                "altura_cm": 170 + (i % 3),
                "sexo": "M" if i % 2 == 0 else "F",
                "idade": 20 + i,
            }
            response = client.post("/users", json=payload)
            assert response.status_code == 200

            data = response.json()
            assert data["name"] == f"User {i}", "The name of the user should match"
            assert data["id"], "The user should have an id"
            assert data["sexo"] in {"M", "F"}, "Sexo deveria ser M ou F"
            assert "imc" in data, "IMC deveria estar presente na resposta"
            assert isinstance(data["imc"], (int, float)), "IMC deveria ser numérico"

            user = User.model_validate({k: v for k, v in data.items() if k != 'imc'})  # Valida o retorno novamente pelo Pydantic
            assert str(user.id) == data["id"], "The id should be the same"
            assert user.signup_ts, "The signup timestamp should be set"
            assert 20 <= user.peso_kg <= 400, "Peso deve ser um valor plausível"
            assert 100 <= user.altura_cm <= 250, "altura deve ser um valor plausível"
            assert 10 <= user.idade <= 120, "idade deve ser um valor plausível"
            assert 10 <= user.imc <= 80, "IMC deve ser um valor plausível"

        response = client.get("/users")
        assert response.status_code == 200, "Response code should be 200"
        assert len(response.json()) == 5, "There should be 5 users"

        # Cria um novo usuário manualmente
        response = client.post(
            "/users",
            json={
                "name": "User 5",
                "email": "example5@arjancodes.com",
                "peso_kg": 82.345,
                "altura_cm": 180,
                "sexo": "f",
                "idade": 33,
            },
        )
        assert response.status_code == 200
        data = response.json()
        assert data["name"] == "User 5"
        assert data["sexo"] == "F", "Sexo deve estar normalizado"
        assert data["peso_kg"] == 82.3, "Peso deve ser arrendondado para uma casa decimal no JSON"
        assert "imc" in data

        user = User.model_validate({k: v for k, v in data.items() if k != 'imc'})
        assert str(user.id) == data["id"], "The id should be the same"
        assert user.signup_ts, "The signup timestamp should be set"

        # Busca usuário pelo id
        response = client.get(f"/users/{data['id']}")
        assert response.status_code == 200
        assert response.json()["name"] == "User 5", "This should be the newly created user"

        # UUID aleatório que não existe
        response = client.get(f"/users/{uuid4()}")
        assert response.status_code == 404
        assert response.json()["message"] == "User not found"

        response = client.post( # Criação de um usuário com email inválido
            "/users",
            json={
                "name": "User 6",
                "email": "wrong",
                "peso_kg": 80,
                "altura_cm": 180,
                "sexo": "M",
                "idade": 30,
            },
        )
        assert response.status_code == 422, "The email address should be invalid"

        response = client.post( # Criação de um usuário com sexo inválido
            "/users",
            json={
                "name": "User 7",
                "email": "example7@arjancodes.com",
                "peso_kg": 80,
                "altura_cm": 180,
                "sexo": "X",
                "idade": 30,
            },
        )
        assert response.status_code == 422, "Sexo inválido"

        response = client.post( # Criação de um usuário com peso fora do intervalo definido
            "/users",
            json={
                "name": "User 8",
                "email": "example8@arjancodes.com",
                "peso_kg": 5,
                "altura_cm": 180,
                "sexo": "M",
                "idade": 30,
            },
        )
        assert response.status_code == 422, "Peso inválido"


In [20]:
if __name__ == "__main__":
    main()